# RF-DETR Large 74-Class Hidden45 Local MPS 5-Fold Training (v1.8+)

## Goal

Train RF-DETR Large on the latest 74-class canvas-balanced pill dataset on local Apple Silicon MPS.

Default path:

```text
local extracted dataset
  -> local 5-fold COCO dataset with class-0 placeholder
  -> RF-DETR train/valid dry-run
  -> local MPS full train/resume
  -> mAP@0.75 checkpoint selection
```

This notebook is the local-MPS counterpart of the Colab CUDA 5-fold notebook. It keeps the same data split, augmentation, 100-epoch cap, and early stopping semantics, but removes Colab Drive/clone/download requirements and writes checkpoints under the local `working/rfdetr_outputs` tree.


## Local MPS Checklist

1. Open this notebook from `/Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver`.
2. Use the local DataAnalysis Python kernel if available: `/Users/pio/.DataAnalysis/bin/python`.
3. The dataset should already exist at `/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45_canvas_balanced`.
4. Run once with `RUN_DRY_RUN=True` and `RUN_FULL_TRAIN=False`; this builds/validates all fold configs without training.
5. Current local setting trains remaining folds with `FOLD_INDICES = [1, 2, 3, 4]`. Fold 0 has already been run; set `[0, 1, 2, 3, 4]` for a full rerun.
6. Local MPS uses `PYTORCH_ENABLE_MPS_FALLBACK=1`; unsupported MPS ops may fall back to CPU and slow down, but this avoids hard failures.
7. If memory becomes unstable, try `MODEL_VARIANT = 'medium'` or reduce augmentation before lowering dataset quality.
8. Test submission/postprocess is deliberately left to the local inference notebook after checkpoints are ready.


In [47]:
# =========================
# Parameters - edit here
# =========================
from pathlib import Path

RUN_MOUNT_DRIVE = False
RUN_CLONE_OR_UPDATE_REPO = False
RUN_INSTALL_DEPS = False
RUN_EXTRACT_DATASET = False
FORCE_REEXTRACT_DATASET = False
USE_READY_DATASET_DIR_IF_EXISTS = True
CACHE_ARCHIVE_BEFORE_EXTRACT = False
RUN_BUILD_5FOLD_DATASET = True
FORCE_REBUILD_5FOLD_DATASET = False
RUN_DRY_RUN = False
RUN_IMPORT_SMOKE_TEST = True
RUN_FULL_TRAIN = True
RUN_FINALIZE_ONLY = False
RUN_SHOW_TENSORBOARD = True
RUN_TEST_INFERENCE = True
REQUIRE_DRIVE_FOR_CHECKPOINTS = False

RFDETR_MIN_VERSION = '1.8.0'

TEAM_REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
TEAM_REPO_BRANCH = 'model/rfdetr-basic74-45img-filled'

PROJECT_ROOT = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject')
WORK_ROOT = PROJECT_ROOT / 'working' / 'rfdetr_mps_large_v18_5fold'
REPO_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr')
RFDETR_DIR = REPO_DIR / 'RF_DETR_split_ver'

# Local dataset paths. No Drive archive is needed for this notebook.
DRIVE_PROJECT_DIR = PROJECT_ROOT / 'working'
DATASET_ARCHIVE_FOLDER_URL = ''
DATASET_ARCHIVE_FILE_ID = ''
DATASET_ARCHIVE_FILE_URL = ''
DATASET_ARCHIVE_DIR = WORK_ROOT / 'archives'
DATASET_ARCHIVE_NAME = 'dataset_74_hidden45_canvas_balanced_train_valid_20260706.tar.gz'
SCAN_MOUNTED_DRIVE_FOR_ARCHIVE = False
ARCHIVE_CACHE_DIR = WORK_ROOT / 'archives'
DATASET_EXTRACT_DIR = WORK_ROOT / 'datasets'
DATASET_DIR = PROJECT_ROOT / 'working' / 'rfdetr_dataset_74_hidden45_canvas_balanced'
FOLDED_DATASET_DIR = PROJECT_ROOT / 'working' / 'rfdetr_dataset_74_hidden45_canvas_balanced_5fold_cls0_mps'

CONFIG_STEM = 'config_74_hidden45_mps_rfdetr_large_v18_5fold'
BASE_CONFIG_NAME = 'config_74_hidden45_canvas_balanced.yaml'
MODEL_VARIANT = 'large'  # nano | small | medium | base | large
DEVICE = 'mps'

# Local resource default: build all 5 folds, but train only fold 0 unless you expand this list.
NUM_FOLDS = 5
FOLD_INDICES = [1, 2, 3, 4]
# Optional explicit resume checkpoints. Fold 0 is pointed at the Colab CUDA checkpoint_4 by default.
# Set a fold value to None to use local auto_resume from OUTPUT_ROOT instead.
RESUME_CHECKPOINT_BY_FOLD = {
    0: Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/l_5f_resume_checkpoints/checkpoint_4.ckpt'),
}
ADD_CLASS0_PLACEHOLDER = True
CLASS0_NAME = 'background_dummy_placeholder'
VALID_INCLUDE_CANVAS = False  # keep synthetic canvas images out of validation; train may still use them

BASE_MODEL_TAG = 'large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus'
ENABLE_TRAIN_AUGMENTATION = True
AUGMENTATION_NAME = 'aug_scale150_rot90_v1'
MODEL_TAG = f'{BASE_MODEL_TAG}_{AUGMENTATION_NAME}' if ENABLE_TRAIN_AUGMENTATION else BASE_MODEL_TAG

# Same effective batch as the CUDA run: 1 x 16 = 16. Large starts at micro-batch 1 for OOM safety.
# Early stopping is validation/eval based, not mini-batch based: eval_interval=1 means patience counts epochs.
# One epoch is roughly 2.3k train images per fold, so patience=1 is the closest match to a ~2k-image no-improvement window.
EPOCHS = 100
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 1
EARLY_STOPPING_MIN_DELTA = 0.001
EARLY_STOPPING_USE_EMA = True
NUM_WORKERS = 2
PIN_MEMORY = False
TRAIN_EPOCHS_OVERRIDE = None  # keep None for config-controlled epochs

# Online train-time augmentation. Flip is intentionally excluded because mirrored pill imprints are unrealistic.
AUG_SCALE_RANGE = [0.85, 1.50]
AUG_TRANSLATE_PERCENT = [-0.04, 0.04]
AUG_ROTATE_LIMIT_DEGREES = 90
AUG_AFFINE_P = 0.35
AUG_BRIGHTNESS_LIMIT = 0.08
AUG_CONTRAST_LIMIT = 0.08
AUG_COLOR_P = 0.25
ENABLE_LIGHT_AUGMENTATION = ENABLE_TRAIN_AUGMENTATION
ENABLE_MULTI_SCALE = False

OUTPUT_ROOT = PROJECT_ROOT / 'working' / 'rfdetr_outputs' / 'mps_large_v18_5fold'
BACKUP_DIR = OUTPUT_ROOT / 'best'


In [48]:
# Imports and small helpers
import json
import os
import random
import selectors
import shutil
import subprocess
import sys
import tarfile
import time
from collections import Counter
from pathlib import Path

import pandas as pd

os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ.setdefault('PYTHONUNBUFFERED', '1')
os.environ.setdefault('PYTHONFAULTHANDLER', '1')
print('PYTORCH_ENABLE_MPS_FALLBACK:', os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK'))


def _print_training_artifact_status():
    output_root = globals().get('OUTPUT_ROOT')
    model_tag = globals().get('MODEL_TAG')
    if output_root is None or model_tag is None:
        print('[status] OUTPUT_ROOT/MODEL_TAG not defined yet.', flush=True)
        return

    output_dir = Path(output_root) / str(model_tag)
    print(f'[status] output_dir={output_dir} exists={output_dir.exists()}', flush=True)
    if not output_dir.exists():
        return

    metrics_csv = output_dir / 'metrics.csv'
    if metrics_csv.exists():
        try:
            tail = metrics_csv.read_text(encoding='utf-8', errors='replace').strip().splitlines()[-3:]
            print('[status] metrics.csv tail:', flush=True)
            for line in tail:
                print('  ' + line, flush=True)
        except Exception as exc:
            print(f'[status] could not read metrics.csv: {exc}', flush=True)
    else:
        print('[status] metrics.csv not created yet.', flush=True)

    checkpoints = sorted(output_dir.glob('checkpoint_*'), key=lambda path: path.stat().st_mtime if path.exists() else 0)
    if checkpoints:
        latest = checkpoints[-1]
        print(f'[status] latest checkpoint={latest.name} size_MB={latest.stat().st_size / (1024 * 1024):.1f}', flush=True)
    else:
        print('[status] no checkpoint files yet.', flush=True)


def run(cmd, cwd=None, env=None, heartbeat_seconds=30):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    merged_env.setdefault('PYTHONUNBUFFERED', '1')
    merged_env.setdefault('PYTHONFAULTHANDLER', '1')
    merged_env.setdefault('WANDB_DISABLED', 'true')
    merged_env.setdefault('WANDB_MODE', 'disabled')

    cmd_list = list(map(str, cmd))
    printable_cmd = ' '.join(cmd_list)
    print('run:', printable_cmd, flush=True)
    print('cwd:', cwd or Path.cwd(), flush=True)
    print('started_at:', time.strftime('%Y-%m-%d %H:%M:%S'), flush=True)

    process = subprocess.Popen(
        cmd_list,
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    selector = selectors.DefaultSelector()
    selector.register(process.stdout, selectors.EVENT_READ)
    start_time = time.monotonic()
    last_output = start_time

    try:
        while True:
            events = selector.select(timeout=5)
            for key, _ in events:
                chunk = os.read(key.fileobj.fileno(), 8192)
                if not chunk:
                    continue
                print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
                last_output = time.monotonic()

            returncode = process.poll()
            if returncode is not None:
                while True:
                    chunk = process.stdout.read(8192)
                    if not chunk:
                        break
                    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
                break

            now = time.monotonic()
            if now - last_output >= heartbeat_seconds:
                elapsed = int(now - start_time)
                print('\n' + f'[still running] elapsed={elapsed}s, no subprocess output for {int(now - last_output)}s', flush=True)
                _print_training_artifact_status()
                last_output = now
    finally:
        try:
            selector.unregister(process.stdout)
        except Exception:
            pass
        if process.stdout:
            process.stdout.close()

    print('finished_at:', time.strftime('%Y-%m-%d %H:%M:%S'), 'returncode:', returncode, flush=True)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd_list)
    return subprocess.CompletedProcess(cmd_list, returncode)


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


seed_everything(42)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT:', WORK_ROOT)
print('Python:', sys.executable)


PYTORCH_ENABLE_MPS_FALLBACK: 1
WORK_ROOT: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_mps_large_v18_5fold
Python: /Users/pio/.DataAnalysis/bin/python


In [49]:
# Local checkpoint persistence check
print('Local MPS notebook: Drive mount is not required.')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print('checkpoint/output root:', OUTPUT_ROOT)
print('backup root:', BACKUP_DIR)


Local MPS notebook: Drive mount is not required.
checkpoint/output root: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold
backup root: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/best


In [50]:
# Use local team repo
if RUN_CLONE_OR_UPDATE_REPO:
    if not REPO_DIR.exists():
        run(['git', 'clone', TEAM_REPO_URL, REPO_DIR])
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    checkout_cmd = ['git', 'checkout', TEAM_REPO_BRANCH]
    try:
        run(checkout_cmd, cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        run(['git', 'checkout', '-B', TEAM_REPO_BRANCH, f'origin/{TEAM_REPO_BRANCH}'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', TEAM_REPO_BRANCH], cwd=REPO_DIR)
else:
    print('Skipping repo clone/update; using local repo:', REPO_DIR)

if not RFDETR_DIR.exists():
    raise FileNotFoundError(f'RF_DETR_split_ver not found: {RFDETR_DIR}')
os.chdir(RFDETR_DIR)
print('cwd:', Path.cwd())


Skipping repo clone/update; using local repo: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr
cwd: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver


In [51]:
# Dependency check / optional install
if RUN_INSTALL_DEPS:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', f'rfdetr[train,loggers]>={RFDETR_MIN_VERSION}', 'packaging'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools', 'faster-coco-eval', 'opencv-python-headless', 'tqdm'])
else:
    print('Skipping dependency install. Current Python:', sys.executable)


Skipping dependency install. Current Python: /Users/pio/.DataAnalysis/bin/python


In [52]:
# Runtime check
import importlib.metadata
import torch
from packaging.version import Version

try:
    import rfdetr
    rfdetr_version = importlib.metadata.version('rfdetr')
    rfdetr_status = f'ok ({rfdetr_version})'
except Exception as exc:
    rfdetr_version = None
    rfdetr_status = f'IMPORT FAILED: {exc}'

mps_available = bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available())
runtime_summary = {
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'mps_available': mps_available,
    'mps_built': bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_built()),
    'device': DEVICE,
    'rfdetr': rfdetr_status,
    'required_rfdetr_min_version': RFDETR_MIN_VERSION,
    'model_variant': MODEL_VARIANT,
    'mps_fallback': os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK'),
}
print(json.dumps(runtime_summary, indent=2, ensure_ascii=False))
if DEVICE == 'mps' and not mps_available:
    raise RuntimeError('MPS is not available in this Python environment. Use the local DataAnalysis/MPS-capable Python kernel.')
if rfdetr_version is None or Version(rfdetr_version) < Version(RFDETR_MIN_VERSION):
    raise RuntimeError(f'rfdetr>={RFDETR_MIN_VERSION} is required, got {rfdetr_version}')
if MODEL_VARIANT == 'large' and not hasattr(rfdetr, 'RFDETRLarge'):
    raise RuntimeError('Installed rfdetr package does not expose RFDETRLarge.')


{
  "torch": "2.12.0",
  "cuda_available": false,
  "mps_available": true,
  "mps_built": true,
  "device": "mps",
  "rfdetr": "ok (1.8.3)",
  "required_rfdetr_min_version": "1.8.0",
  "model_variant": "large",
  "mps_fallback": "1"
}


In [53]:
# Use local extracted dataset; no Drive/archive download

def dataset_ready(dataset_dir: Path) -> bool:
    return (dataset_dir / 'train' / '_annotations.coco.json').exists() and (dataset_dir / 'valid' / '_annotations.coco.json').exists()

ARCHIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if USE_READY_DATASET_DIR_IF_EXISTS and dataset_ready(DATASET_DIR):
    archive_path = None
    archive_source = 'skipped: local DATASET_DIR is already ready'
    print('Using ready local dataset:', DATASET_DIR)
else:
    raise FileNotFoundError(
        f'DATASET_DIR is not ready: {DATASET_DIR}. '
        'Expected train/valid/_annotations.coco.json under the local canvas-balanced dataset.'
    )

if not dataset_ready(DATASET_DIR):
    raise FileNotFoundError(f'Dataset is not ready: {DATASET_DIR}')


Using ready local dataset: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45_canvas_balanced


In [54]:
# Dataset sanity checks
train_json = DATASET_DIR / 'train' / '_annotations.coco.json'
valid_json = DATASET_DIR / 'valid' / '_annotations.coco.json'
if not train_json.exists() or not valid_json.exists():
    raise FileNotFoundError(f'Missing train/valid annotations under {DATASET_DIR}')

summary_path = DATASET_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('dataset summary keys:', list(summary.keys()))
    if 'splits' in summary:
        display(pd.DataFrame(summary['splits']))
    if 'validation' in summary:
        print(json.dumps(summary['validation'].get('train_valid', summary['validation']), indent=2, ensure_ascii=False))
else:
    print('No summary.json found; continuing with COCO JSON checks.')

def load_coco(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

splits = {'train': load_coco(train_json), 'valid': load_coco(valid_json)}
rows = []
for split, data in splits.items():
    category_ids = sorted(int(c['id']) for c in data['categories'])
    ann_category_ids = sorted({int(a['category_id']) for a in data['annotations']})
    missing_files = []
    for image in data['images'][:]:
        if not (DATASET_DIR / split / image['file_name']).exists():
            missing_files.append(image['file_name'])
    rows.append({
        'split': split,
        'images': len(data['images']),
        'annotations': len(data['annotations']),
        'categories': len(category_ids),
        'ann_categories': len(ann_category_ids),
        'min_category_id': min(category_ids),
        'max_category_id': max(category_ids),
        'missing_files': len(missing_files),
    })
    if missing_files:
        raise FileNotFoundError(f'{split} has missing files, sample={missing_files[:5]}')

display(pd.DataFrame(rows))
category_counts = Counter(a['category_id'] for data in splits.values() for a in data['annotations'])
print('class annotation range:', min(category_counts.values()), 'to', max(category_counts.values()))
print('classes:', len(category_counts))


dataset summary keys: ['input_dir', 'out_dir', 'seed', 'canvas_size', 'valid_split', 'canvas', 'splits', 'validation', 'files']


,split,images,annotations,canvas_images,canvas_annotations,categories
0,train,2541,2962,1174.0,1174.0,NaN
1,valid,362,480,0.0,0.0,NaN
2,test,842,0,NaN,NaN,74.0


{
  "split_group_leakage_count": 0,
  "class_count": 74,
  "total_min_class_annotations": 45,
  "total_max_class_annotations": 157,
  "valid_min_class_annotations": 6,
  "valid_max_class_annotations": 32,
  "classes_with_zero_train": 0,
  "classes_with_zero_valid": 0
}


,split,images,annotations,categories,ann_categories,min_category_id,max_category_id,missing_files
0,train,2541,2962,74,74,1900,44199,0
1,valid,362,480,74,74,1900,44199,0


class annotation range: 45 to 157
classes: 74


In [55]:
# Build 5-fold COCO dataset with class-0 dummy placeholder
if RUN_BUILD_5FOLD_DATASET:
    import csv
    import json
    import os
    import random
    import re
    import shutil
    from collections import Counter, defaultdict
    from pathlib import Path

    import numpy as np
    from sklearn.model_selection import StratifiedGroupKFold

    ready_marker = FOLDED_DATASET_DIR / '_5fold_ready.json'
    if ready_marker.exists() and not FORCE_REBUILD_5FOLD_DATASET:
        print('5-fold dataset already exists:', FOLDED_DATASET_DIR)
        print(ready_marker.read_text(encoding='utf-8'))
    else:
        if FOLDED_DATASET_DIR.exists():
            shutil.rmtree(FOLDED_DATASET_DIR)
        FOLDED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

        def load_split(split):
            ann_path = DATASET_DIR / split / '_annotations.coco.json'
            payload = json.loads(ann_path.read_text(encoding='utf-8'))
            images_by_id = {int(img['id']): img for img in payload.get('images', [])}
            anns_by_image = defaultdict(list)
            for ann in payload.get('annotations', []):
                anns_by_image[int(ann['image_id'])].append(ann)
            return payload, images_by_id, anns_by_image

        train_payload, train_images, train_anns = load_split('train')
        valid_payload, valid_images, valid_anns = load_split('valid')
        source_categories = {int(cat['id']): cat for cat in train_payload.get('categories', [])}
        source_categories.update({int(cat['id']): cat for cat in valid_payload.get('categories', [])})
        real_category_ids = sorted(cid for cid in source_categories if cid != 0)
        cat2label = {cid: i + 1 for i, cid in enumerate(real_category_ids)}
        label2cat = {label: cid for cid, label in cat2label.items()}

        def normalize_group_name(file_name):
            stem = Path(file_name).stem
            core = stem
            if core.startswith('canvas__'):
                core = core[len('canvas__'):]
            if '__' in core:
                core = core.split('__', 1)[1]
            match = re.search(r'(K-[0-9-]+_0_2_\d_2)_(\d+)(_000_200)', core)
            if match:
                return f'{match.group(1)}_ANGLE{match.group(3)}'
            return re.sub(r'_(70|75|90)_000_200', '_ANGLE_000_200', core)

        records = []
        used_output_names = Counter()
        for split, images_by_id, anns_by_image in [('train', train_images, train_anns), ('valid', valid_images, valid_anns)]:
            for image_id, image in images_by_id.items():
                anns = anns_by_image.get(image_id, [])
                if not anns:
                    continue
                src_path = DATASET_DIR / split / image['file_name']
                if not src_path.exists():
                    raise FileNotFoundError(src_path)
                base_output_name = f'{split}__{image["file_name"]}'
                used_output_names[base_output_name] += 1
                output_name = base_output_name
                if used_output_names[base_output_name] > 1:
                    output_name = f'{split}__dup{used_output_names[base_output_name]}__{image["file_name"]}'
                labels = sorted({int(ann['category_id']) for ann in anns})
                records.append({
                    'source_split': split,
                    'source_image_id': image_id,
                    'source_file_name': image['file_name'],
                    'output_file_name': output_name,
                    'width': int(image['width']),
                    'height': int(image['height']),
                    'src_path': src_path,
                    'is_canvas': image['file_name'].startswith('canvas__'),
                    'annotations': anns,
                    'category_ids': labels,
                    'group': normalize_group_name(image['file_name']),
                })

        if not records:
            raise RuntimeError('No annotated images found for 5-fold build.')

        freq = Counter(cid for rec in records for cid in rec['category_ids'])
        valid_candidate_indices = [
            i for i, rec in enumerate(records)
            if VALID_INCLUDE_CANVAS or not rec['is_canvas']
        ]
        if len(valid_candidate_indices) < NUM_FOLDS:
            raise RuntimeError(f'Not enough validation candidates for {NUM_FOLDS} folds: {len(valid_candidate_indices)}')

        file_indices = np.array(valid_candidate_indices)
        groups = np.array([records[i]['group'] for i in valid_candidate_indices])
        strat = np.array([min(records[i]['category_ids'], key=lambda cid: freq[cid]) for i in valid_candidate_indices])

        sgkf = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
        folds = list(sgkf.split(file_indices, strat, groups))
        print('5-fold validation candidates:', len(valid_candidate_indices), 'of records:', len(records), 'VALID_INCLUDE_CANVAS=', VALID_INCLUDE_CANVAS)

        def link_or_copy(src, dst):
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            try:
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)

        def categories_for_fold():
            cats = []
            if ADD_CLASS0_PLACEHOLDER:
                cats.append({
                    'id': 0,
                    'name': CLASS0_NAME,
                    'supercategory': 'background',
                    'is_placeholder': True,
                    'used_in_annotations': False,
                    'note': 'Background/no-object dummy category. Keep bbox annotations at zero; never submit category 0.',
                })
            for cid in real_category_ids:
                src_cat = dict(source_categories[cid])
                label = cat2label[cid]
                cats.append({
                    'id': label,
                    'name': str(src_cat.get('name', cid)),
                    'supercategory': src_cat.get('supercategory', 'pill'),
                    'original_category_id': cid,
                    'source_name': src_cat.get('name', ''),
                })
            return cats

        fold_summaries = []
        category_rows = []
        if ADD_CLASS0_PLACEHOLDER:
            category_rows.append({
                'rfdetr_internal_label': 0,
                'coco_category_id': 0,
                'category_id': 0,
                'n_number': 'N00',
                'name': CLASS0_NAME,
                'is_placeholder': True,
            })
        source_mapping_path = DATASET_DIR / 'category_mapping.csv'
        source_mapping = {}
        if source_mapping_path.exists():
            with source_mapping_path.open('r', encoding='utf-8-sig', newline='') as f:
                for row in csv.DictReader(f):
                    source_mapping[int(row['category_id'])] = row
        for cid in real_category_ids:
            src_row = source_mapping.get(cid, {})
            category_rows.append({
                'rfdetr_internal_label': cat2label[cid],
                'coco_category_id': cat2label[cid],
                'category_id': cid,
                'n_number': src_row.get('n_number', ''),
                'name': src_row.get('name', source_categories[cid].get('name', '')),
                'is_placeholder': False,
            })

        all_indices = list(range(len(records)))
        for fold_idx, (_candidate_train_idx, candidate_valid_idx) in enumerate(folds):
            fold_dir = FOLDED_DATASET_DIR / f'fold{fold_idx}'
            valid_idx = [valid_candidate_indices[int(i)] for i in candidate_valid_idx]
            valid_groups = {records[i]['group'] for i in valid_idx}
            train_idx = [i for i in all_indices if records[i]['group'] not in valid_groups]
            train_groups = {records[i]['group'] for i in train_idx}
            leakage = sorted(train_groups & valid_groups)
            if leakage:
                raise RuntimeError(f'fold{fold_idx} group leakage detected: {leakage[:5]}')
            if not VALID_INCLUDE_CANVAS and any(records[i]['is_canvas'] for i in valid_idx):
                raise RuntimeError(f'fold{fold_idx} validation unexpectedly contains canvas images')

            for split_name, indices in [('train', train_idx), ('valid', valid_idx)]:
                split_dir = fold_dir / split_name
                split_dir.mkdir(parents=True, exist_ok=True)
                images = []
                annotations = []
                ann_id = 1
                for new_image_id, rec_index in enumerate(indices, start=1):
                    rec = records[int(rec_index)]
                    images.append({
                        'id': new_image_id,
                        'file_name': rec['output_file_name'],
                        'width': rec['width'],
                        'height': rec['height'],
                        'source_split': rec['source_split'],
                        'source_file_name': rec['source_file_name'],
                        'group_key': rec['group'],
                    })
                    link_or_copy(rec['src_path'], split_dir / rec['output_file_name'])
                    for ann in rec['annotations']:
                        original_cid = int(ann['category_id'])
                        if original_cid == 0:
                            raise RuntimeError('source annotation unexpectedly used category_id=0')
                        bbox = [float(v) for v in ann['bbox']]
                        annotations.append({
                            'id': ann_id,
                            'image_id': new_image_id,
                            'category_id': cat2label[original_cid],
                            'bbox': bbox,
                            'area': float(ann.get('area', bbox[2] * bbox[3])),
                            'iscrowd': int(ann.get('iscrowd', 0)),
                            'original_category_id': original_cid,
                            'source_annotation_id': ann.get('id', ''),
                        })
                        ann_id += 1
                payload = {
                    'info': {
                        'description': 'RF-DETR 74-class 5-fold dataset with class-0 background/no-object dummy category',
                        'class0_semantics': 'category_id=0 is the background/no-object dummy category; bbox annotations use labels 1..74.',
                    },
                    'images': images,
                    'annotations': annotations,
                    'categories': categories_for_fold(),
                }
                (split_dir / '_annotations.coco.json').write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')

            train_class_counts = Counter(
                cid for i in train_idx for cid in records[i]['category_ids']
            )
            valid_class_counts = Counter(
                cid for i in valid_idx for cid in records[i]['category_ids']
            )
            fold_summary = {
                'fold': fold_idx,
                'train_images': int(len(train_idx)),
                'valid_images': int(len(valid_idx)),
                'train_canvas_images': int(sum(records[i]['is_canvas'] for i in train_idx)),
                'valid_canvas_images': int(sum(records[i]['is_canvas'] for i in valid_idx)),
                'train_groups': int(len(train_groups)),
                'valid_groups': int(len(valid_groups)),
                'group_leakage': 0,
                'train_classes': int(len(train_class_counts)),
                'valid_classes': int(len(valid_class_counts)),
                'valid_min_class_annotations': int(min(valid_class_counts.values())) if valid_class_counts else 0,
                'valid_max_class_annotations': int(max(valid_class_counts.values())) if valid_class_counts else 0,
            }
            fold_summaries.append(fold_summary)
            print('fold summary:', fold_summary)

        label_map = {
            'class0': {
                'id': 0,
                'name': CLASS0_NAME,
                'is_placeholder': True,
                'used_in_annotations': False,
                'purpose': 'Represent the background/no-object dummy category; real pill classes use labels 1..74.',
                'submission_rule': 'Drop predictions mapped to category_id 0 during local postprocess.',
            },
            'cat2label': {str(k): int(v) for k, v in cat2label.items()},
            'label2cat': {str(k): int(v) for k, v in label2cat.items()},
            'num_real_classes': len(real_category_ids),
            'num_coco_categories_including_class0': len(real_category_ids) + int(ADD_CLASS0_PLACEHOLDER),
        }
        (FOLDED_DATASET_DIR / 'label_map_74.json').write_text(json.dumps(label_map, ensure_ascii=False, indent=2), encoding='utf-8')

        with (FOLDED_DATASET_DIR / 'category_mapping.csv').open('w', encoding='utf-8-sig', newline='') as f:
            fieldnames = ['rfdetr_internal_label', 'coco_category_id', 'category_id', 'n_number', 'name', 'is_placeholder']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(category_rows)

        # Validate class 0 is never used in annotations.
        class0_annotation_count = 0
        for path in FOLDED_DATASET_DIR.glob('fold*/**/_annotations.coco.json'):
            payload = json.loads(path.read_text(encoding='utf-8'))
            class0_annotation_count += sum(1 for ann in payload.get('annotations', []) if int(ann['category_id']) == 0)
        if class0_annotation_count != 0:
            raise RuntimeError(f'class0 annotation count must be 0, got {class0_annotation_count}')

        ready = {
            'folds': fold_summaries,
            'records': len(records),
            'real_classes': len(real_category_ids),
            'class0_placeholder': bool(ADD_CLASS0_PLACEHOLDER),
            'valid_include_canvas': bool(VALID_INCLUDE_CANVAS),
            'class0_annotation_count': class0_annotation_count,
            'label_map': str(FOLDED_DATASET_DIR / 'label_map_74.json'),
            'category_mapping': str(FOLDED_DATASET_DIR / 'category_mapping.csv'),
        }
        ready_marker.write_text(json.dumps(ready, ensure_ascii=False, indent=2), encoding='utf-8')
        print(json.dumps(ready, ensure_ascii=False, indent=2))
else:
    print('Skipping 5-fold dataset build. Using existing:', FOLDED_DATASET_DIR)


5-fold dataset already exists: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45_canvas_balanced_5fold_cls0_mps
{
  "folds": [
    {
      "fold": 0,
      "train_images": 2337,
      "valid_images": 346,
      "train_canvas_images": 954,
      "valid_canvas_images": 0,
      "train_groups": 600,
      "valid_groups": 135,
      "group_leakage": 0,
      "train_classes": 74,
      "valid_classes": 74,
      "valid_min_class_annotations": 3,
      "valid_max_class_annotations": 29
    },
    {
      "fold": 1,
      "train_images": 2345,
      "valid_images": 347,
      "train_canvas_images": 963,
      "valid_canvas_images": 0,
      "train_groups": 597,
      "valid_groups": 138,
      "group_leakage": 0,
      "train_classes": 74,
      "valid_classes": 74,
      "valid_min_class_annotations": 3,
      "valid_max_class_annotations": 29
    },
    {
      "fold": 2,
      "train_images": 2339,
      "valid_images": 346,
      "train_canvas_image

In [56]:
# Create per-fold local MPS configs from the repo config
import copy
import yaml

base_config_path = RFDETR_DIR / BASE_CONFIG_NAME
if not base_config_path.exists():
    base_config_path = RFDETR_DIR / 'config_74_hidden45.yaml'
print('base config:', base_config_path)

base_config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))

TRAIN_AUG_CONFIG = {
    'RandomBrightnessContrast': {
        'brightness_limit': float(AUG_BRIGHTNESS_LIMIT),
        'contrast_limit': float(AUG_CONTRAST_LIMIT),
        'p': float(AUG_COLOR_P),
    },
    'Affine': {
        'scale': [float(AUG_SCALE_RANGE[0]), float(AUG_SCALE_RANGE[1])],
        'keep_ratio': True,
        'translate_percent': [float(AUG_TRANSLATE_PERCENT[0]), float(AUG_TRANSLATE_PERCENT[1])],
        'rotate': [-float(AUG_ROTATE_LIMIT_DEGREES), float(AUG_ROTATE_LIMIT_DEGREES)],
        'shear': 0,
        'border_mode': 4,
        'p': float(AUG_AFFINE_P),
    },
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
fold_config_paths = []
fold_model_tags = []

for fold_idx in FOLD_INDICES:
    fold_dir = FOLDED_DATASET_DIR / f'fold{fold_idx}'
    if not (fold_dir / 'train' / '_annotations.coco.json').exists():
        raise FileNotFoundError(f'fold train annotations missing: {fold_dir}')
    if not (fold_dir / 'valid' / '_annotations.coco.json').exists():
        raise FileNotFoundError(f'fold valid annotations missing: {fold_dir}')

    config = copy.deepcopy(base_config)
    config['data']['dataset_dir'] = str(fold_dir)
    config['data']['base56_dir'] = ''
    config['data']['hidden18_dir'] = ''
    config['data']['test_images_dir'] = ''

    fold_tag = f'{MODEL_TAG}_p{fold_idx:02d}'
    config['model']['variant'] = MODEL_VARIANT
    config['model']['tag'] = fold_tag

    train_cfg = config['train']
    explicit_resume = RESUME_CHECKPOINT_BY_FOLD.get(int(fold_idx)) if 'RESUME_CHECKPOINT_BY_FOLD' in globals() else None
    if explicit_resume is not None:
        explicit_resume = Path(explicit_resume).expanduser()
        if not explicit_resume.exists():
            raise FileNotFoundError(f'explicit resume checkpoint for fold {fold_idx} not found: {explicit_resume}')
        print('explicit resume checkpoint:', explicit_resume)
    train_cfg.update({
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'seed': 42 + int(fold_idx),
        'device': DEVICE,
        'pin_memory': PIN_MEMORY,
        'num_workers': NUM_WORKERS,
        'persistent_workers': bool(NUM_WORKERS > 0),
        'prefetch_factor': 2 if NUM_WORKERS > 0 else None,
        'checkpoint_interval': 1,
        'auto_resume': True,
        'resume': str(explicit_resume) if explicit_resume is not None else None,
        'eval_interval': 1,
        'early_stopping': EARLY_STOPPING,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta': EARLY_STOPPING_MIN_DELTA,
        'early_stopping_use_ema': EARLY_STOPPING_USE_EMA,
        'eval_max_dets': 100,
        'compute_val_loss': True,
        'compute_test_loss': False,
        'run_test': False,
        'progress_bar': 'tqdm',
        'tensorboard': True,
        'multi_scale': ENABLE_MULTI_SCALE,
        'expanded_scales': False,
        'amp_dtype': 'auto',
    })
    if ENABLE_TRAIN_AUGMENTATION:
        train_cfg['aug_config'] = TRAIN_AUG_CONFIG
        train_cfg['augmentation_backend'] = 'cpu'
    else:
        train_cfg.pop('aug_config', None)
        train_cfg['augmentation_backend'] = 'cpu'

    config['output']['local_output_dir'] = str(OUTPUT_ROOT)
    config['output']['backup_dir'] = str(BACKUP_DIR)

    config_path = RFDETR_DIR / f'{CONFIG_STEM}_p{fold_idx:02d}.yaml'
    config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
    fold_config_paths.append(config_path)
    fold_model_tags.append(fold_tag)
    print('wrote config:', config_path)
    print('fold tag:', fold_tag)
    print('fold dataset:', fold_dir)

print('model variant:', MODEL_VARIANT)
print('rfdetr min version:', RFDETR_MIN_VERSION)
print('folds:', FOLD_INDICES)
print('train augmentation enabled:', ENABLE_TRAIN_AUGMENTATION)
if ENABLE_TRAIN_AUGMENTATION:
    print(yaml.safe_dump(TRAIN_AUG_CONFIG, allow_unicode=True, sort_keys=False))
print('checkpoints will be saved under:', OUTPUT_ROOT / '<fold_model_tag>')
print('best checkpoint backups will be saved under:', BACKUP_DIR)


base config: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45_canvas_balanced.yaml
wrote config: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45_mps_rfdetr_large_v18_5fold_p01.yaml
fold tag: large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p01
fold dataset: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45_canvas_balanced_5fold_cls0_mps/fold1
wrote config: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45_mps_rfdetr_large_v18_5fold_p02.yaml
fold tag: large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p02
fold dataset: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45_canvas_balanced_5fold_cls0_mps/fold2
wrote config: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/c

In [57]:
# Preflight: validates imports, config, dataset files, and train kwargs without training
if RUN_IMPORT_SMOKE_TEST:
    smoke_code = f"""
import importlib.metadata
import sys
from packaging.version import Version
print('smoke: python', sys.executable, flush=True)
import torch
print('smoke: torch', torch.__version__, 'cuda_available=', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('smoke: cuda device', torch.cuda.get_device_name(0), flush=True)
import rfdetr
version = importlib.metadata.version('rfdetr')
print('smoke: rfdetr', version, 'RFDETRLarge=', hasattr(rfdetr, 'RFDETRLarge'), flush=True)
if Version(version) < Version('{RFDETR_MIN_VERSION}'):
    raise RuntimeError('rfdetr version is below required minimum')
if not hasattr(rfdetr, 'RFDETRLarge'):
    raise RuntimeError('RFDETRLarge is unavailable')
""".strip()
    run([sys.executable, '-u', '-c', smoke_code], cwd=RFDETR_DIR, heartbeat_seconds=15)
else:
    print('Skipping import smoke test.')

if RUN_DRY_RUN:
    for config_path in fold_config_paths:
        print('dry-run config:', config_path)
        run([sys.executable, '-u', 'train_74_hidden45.py', '--config', config_path, '--dry-run'], cwd=RFDETR_DIR, heartbeat_seconds=15)
else:
    print('Skipping dry-run.')


run: /Users/pio/.DataAnalysis/bin/python -u -c import importlib.metadata
import sys
from packaging.version import Version
print('smoke: python', sys.executable, flush=True)
import torch
print('smoke: torch', torch.__version__, 'cuda_available=', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('smoke: cuda device', torch.cuda.get_device_name(0), flush=True)
import rfdetr
version = importlib.metadata.version('rfdetr')
print('smoke: rfdetr', version, 'RFDETRLarge=', hasattr(rfdetr, 'RFDETRLarge'), flush=True)
if Version(version) < Version('1.8.0'):
    raise RuntimeError('rfdetr version is below required minimum')
if not hasattr(rfdetr, 'RFDETRLarge'):
    raise RuntimeError('RFDETRLarge is unavailable')
cwd: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver
started_at: 2026-07-07 22:04:11
smoke: python /Users/pio/.DataAnalysis/bin/python
smoke: torch 2.12.0 cuda_available= False
/Users/pio/.DataAnalysis/lib/python3.11/site-packages/

In [58]:
# Full local MPS fold train / resume / finalize
for fold_idx, config_path in zip(FOLD_INDICES, fold_config_paths):
    train_cmd = [sys.executable, '-u', 'train_74_hidden45.py', '--config', config_path]
    if TRAIN_EPOCHS_OVERRIDE is not None:
        train_cmd += ['--epochs', str(TRAIN_EPOCHS_OVERRIDE)]

    finalize_cmd = [sys.executable, '-u', 'train_74_hidden45.py', '--config', config_path, '--finalize-only']
    print('=' * 80)
    print(f'fold {fold_idx}')
    print('train command:', ' '.join(map(str, train_cmd)))
    print('finalize command:', ' '.join(map(str, finalize_cmd)))

    if RUN_FINALIZE_ONLY:
        run(finalize_cmd, cwd=RFDETR_DIR, env={'PYTORCH_ENABLE_MPS_FALLBACK': '1'})
    elif RUN_FULL_TRAIN:
        run(train_cmd, cwd=RFDETR_DIR, env={'PYTORCH_ENABLE_MPS_FALLBACK': '1'})
    else:
        print('RUN_FULL_TRAIN=False. Set RUN_FULL_TRAIN=True in the parameter cell to launch or resume this fold.')


fold 1
train command: /Users/pio/.DataAnalysis/bin/python -u train_74_hidden45.py --config /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45_mps_rfdetr_large_v18_5fold_p01.yaml
finalize command: /Users/pio/.DataAnalysis/bin/python -u train_74_hidden45.py --config /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45_mps_rfdetr_large_v18_5fold_p01.yaml --finalize-only
run: /Users/pio/.DataAnalysis/bin/python -u train_74_hidden45.py --config /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45_mps_rfdetr_large_v18_5fold_p01.yaml
cwd: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver
started_at: 2026-07-07 22:04:16
/Users/pio/.DataAnalysis/lib/python3.11/site-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, 

In [59]:
# Metrics and checkpoints for all selected folds
import json
import pandas as pd

for fold_idx, fold_tag in zip(FOLD_INDICES, fold_model_tags):
    output_dir = OUTPUT_ROOT / fold_tag
    backup_dir = BACKUP_DIR
    metrics_csv = output_dir / 'metrics.csv'
    print('=' * 80)
    print('fold:', fold_idx)
    print('output_dir:', output_dir)
    print('backup_dir:', backup_dir)
    print('metrics_csv:', metrics_csv)

    if output_dir.exists():
        for path in sorted(output_dir.glob('*')):
            size = f'{path.stat().st_size / (1024 * 1024):.1f} MB' if path.is_file() else '<dir>'
            print(path.name, size)
    else:
        print('No output_dir yet.')

    if metrics_csv.exists():
        metrics_df = pd.read_csv(metrics_csv)
        cols = [c for c in ['epoch', 'val/mAP_75', 'val/mAP_50_95', 'val/mAP_50', 'val/mAR', 'val/loss', 'train/loss'] if c in metrics_df.columns]
        per_epoch = metrics_df.groupby('epoch').last().reset_index()[cols]
        display(per_epoch.tail(20))
        if 'val/mAP_75' in per_epoch.columns and per_epoch['val/mAP_75'].notna().any():
            best_idx = per_epoch['val/mAP_75'].idxmax()
            best = per_epoch.loc[best_idx]
            print(f"Best mAP@0.75: epoch={int(best['epoch'])}, score={float(best['val/mAP_75']):.6f}")
            print('best mAP75 checkpoint:', output_dir / 'checkpoint_best_map75.ckpt')
    else:
        print('metrics.csv not found yet. Run training first.')

    map75_summary_path = output_dir / 'map75_summary.json'
    if map75_summary_path.exists():
        print('map75 summary:')
        print(map75_summary_path.read_text(encoding='utf-8'))

label_map_path = FOLDED_DATASET_DIR / 'label_map_74.json'
category_mapping_path = FOLDED_DATASET_DIR / 'category_mapping.csv'
print('label_map_74:', label_map_path, 'exists=', label_map_path.exists())
print('category_mapping:', category_mapping_path, 'exists=', category_mapping_path.exists())


fold: 1
output_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p01
backup_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/best
metrics_csv: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p01/metrics.csv
checkpoint_0.ckpt 509.9 MB
checkpoint_1.ckpt 509.9 MB
checkpoint_2.ckpt 509.9 MB
checkpoint_3.ckpt 509.9 MB
checkpoint_4.ckpt 509.9 MB
checkpoint_5.ckpt 509.9 MB
checkpoint_6.ckpt 509.9 MB
checkpoint_best_ema.pth 127.5 MB
checkpoint_best_map75.ckpt 509.9 MB
checkpoint_best_regular.pth 127.5 MB
checkpoint_best_total.pth 127.4 MB
events.out.tfevents.1783429461.MacBook-Pro.local.30242.0 0.0 MB
hparams.yaml 0.0 MB
map75_summary.json 0.0 MB
metrics.csv 

,epoch,val/mAP_75,val/mAP_50_95,val/mAP_50,val/mAR,val/loss,train/loss
0,0,0.362644,0.341517,0.362644,0.846111,8.274370,9.621423
1,1,0.829511,0.805330,0.829511,0.974361,5.544559,7.649789
2,2,0.938292,0.913043,0.938292,0.978072,3.870702,5.790777
3,3,0.975707,0.949324,0.975707,0.979785,3.434306,4.838722
4,4,0.990663,0.970460,0.990663,0.985956,3.280904,4.729441
5,5,0.997160,0.976797,0.997160,0.985948,3.367188,4.624541
6,6,0.999739,0.971219,0.999739,0.978190,3.473288,4.319171


Best mAP@0.75: epoch=6, score=0.999739
best mAP75 checkpoint: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p01/checkpoint_best_map75.ckpt
map75 summary:
{
  "epoch": 6,
  "step": 1028,
  "map75": 0.9997387528419495,
  "map50_95": 0.9712189435958862,
  "ema_map50_95": 0.9906837940216064,
  "source_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p01/checkpoint_6.ckpt",
  "output_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p01/checkpoint_best_map75.ckpt",
  "backup_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/wor

,epoch,val/mAP_75,val/mAP_50_95,val/mAP_50,val/mAR,val/loss,train/loss
0,0,0.340619,0.319416,0.340619,0.848751,8.218733,9.530990
1,1,0.804454,0.786614,0.804454,0.972320,5.626032,7.759116
2,2,0.927130,0.906721,0.927130,0.979544,4.175367,5.928305
3,3,0.985828,0.968257,0.985828,0.989492,3.034135,4.920963
4,4,0.992682,0.948551,0.992682,0.969776,3.732486,4.540920
5,5,0.993866,0.967927,0.993866,0.982504,2.961727,4.294348
6,6,0.999125,0.975881,0.999125,0.982673,2.725792,3.853963
7,7,1.000000,0.979747,1.000000,0.985622,2.553142,3.939054
8,8,0.999807,0.984787,0.999807,0.988453,2.394351,3.672012
9,9,1.000000,0.986056,1.000000,0.990061,2.566299,3.594413


Best mAP@0.75: epoch=7, score=1.000000
best mAP75 checkpoint: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p02/checkpoint_best_map75.ckpt
map75 summary:
{
  "epoch": 7,
  "step": 1175,
  "map75": 1.0,
  "map50_95": 0.9797466397285461,
  "ema_map50_95": 0.9900717735290527,
  "source_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p02/checkpoint_7.ckpt",
  "output_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p02/checkpoint_best_map75.ckpt",
  "backup_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_out

,epoch,val/mAP_75,val/mAP_50_95,val/mAP_50,val/mAR,val/loss,train/loss
0,0,0.258156,0.244421,0.258156,0.845339,8.122110,9.599377
1,1,0.802287,0.783604,0.802287,0.959754,5.636402,7.752539
2,2,0.930221,0.909489,0.930221,0.982467,4.111604,6.144209
3,3,0.950141,0.926057,0.950141,0.980591,3.797722,5.148923
4,4,0.991637,0.967742,0.991637,0.981609,3.358819,4.691090
5,5,0.995248,0.965718,0.995248,0.978529,3.052925,4.457373
6,6,0.998916,0.960353,0.998916,0.967671,2.975372,4.244836
7,7,0.999735,0.983075,0.999735,0.989231,2.534979,4.206145
8,8,0.999068,0.941151,0.999068,0.954227,3.456849,3.987371
9,9,0.995090,0.981775,0.995090,0.991249,2.380060,3.698085


Best mAP@0.75: epoch=7, score=0.999735
best mAP75 checkpoint: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p03/checkpoint_best_map75.ckpt
map75 summary:
{
  "epoch": 7,
  "step": 1183,
  "map75": 0.9997353553771973,
  "map50_95": 0.9830746650695801,
  "ema_map50_95": 0.9912338256835938,
  "source_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p03/checkpoint_7.ckpt",
  "output_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p03/checkpoint_best_map75.ckpt",
  "backup_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/wor

,epoch,val/mAP_75,val/mAP_50_95,val/mAP_50,val/mAR,val/loss,train/loss
0,0,0.293918,0.278442,0.294125,0.839305,8.366206,9.578048
1,1,0.841096,0.809907,0.847209,0.970771,5.547378,7.609081
2,2,0.944453,0.916719,0.949776,0.976905,3.938405,5.970313
3,3,0.964567,0.939150,0.966672,0.978715,3.574873,4.867437
4,4,0.973124,0.941902,0.977600,0.970455,3.397888,4.764229
5,5,0.989189,0.958294,0.991075,0.976393,3.431617,4.346471
6,6,0.996656,0.975884,0.998541,0.984308,2.699909,4.250453
7,7,0.992778,0.981481,0.998193,0.989490,2.656319,4.068488
8,8,0.997754,0.982682,0.999749,0.986502,2.484196,3.893287
9,9,0.998005,0.976691,1.000000,0.982603,2.693206,3.933643


Best mAP@0.75: epoch=9, score=0.998005
best mAP75 checkpoint: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p04/checkpoint_best_map75.ckpt
map75 summary:
{
  "epoch": 9,
  "step": 1479,
  "map75": 0.998005211353302,
  "map50_95": 0.9766914248466492,
  "ema_map50_95": 0.9901032447814941,
  "source_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p04/checkpoint_9.ckpt",
  "output_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/mps_large_v18_5fold/large_74_hidden45_canvas_balanced_5fold_cls0_local_mps_rfdetr_v18plus_aug_scale150_rot90_v1_p04/checkpoint_best_map75.ckpt",
  "backup_checkpoint": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/work

## Local Submission/Postprocess

This notebook trains local MPS fold checkpoints only.

After training, run the local inference/postprocess notebook or script for:

- checkpoint ensemble or single-checkpoint inference
- category-id mapping back to competition IDs
- threshold/no-overlap postprocess
- low-confidence visual review
- final submission CSV


In [60]:
# No submission in this training notebook
print('Submission/postprocess is intentionally skipped here. Use local inference after checkpoints are ready.')


Submission/postprocess is intentionally skipped here. Use local inference after checkpoints are ready.


In [61]:
# Reserved for local-postprocess notes
print('No-op: final submission CSV is generated locally, not in Colab.')


No-op: final submission CSV is generated locally, not in Colab.


In [62]:
# Reserved for local-postprocess sanity checks
print('No-op: local postprocess should drop category_id=0 and then apply the selected box filtering policy.')


No-op: local postprocess should drop category_id=0 and then apply the selected box filtering policy.


In [63]:
# Optional TensorBoard
if RUN_SHOW_TENSORBOARD:
    ip = get_ipython()
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {OUTPUT_ROOT}')
else:
    print('Skipping TensorBoard. Set RUN_SHOW_TENSORBOARD=True to show it.')


## Augmentation And Class-0 Notes

Train-time augmentation is controlled in the parameter cell:

- scale `[0.85, 1.50]`
- translate `[-0.04, 0.04]`
- rotate `[-90, 90]` degrees via `AUG_ROTATE_LIMIT_DEGREES`
- brightness/contrast `0.08`, probability `0.25`
- no flip, because mirrored pill imprints are unrealistic

Validation folds exclude synthetic `canvas__` images by default (`VALID_INCLUDE_CANVAS=False`), while train folds may still use canvas samples.

Class `0` is included as the background/no-object dummy category. It is not a real pill class and should not receive bbox annotations; RF-DETR/DETR learns background through unmatched queries and non-annotated image regions. The fold builder validates that `category_id=0` has zero annotations. Local inference/postprocess should drop any prediction mapped to category `0`.

Training is capped at `EPOCHS=100`, with early stopping enabled (`patience=1`, `min_delta=0.001`, `use_ema=True`). Because `eval_interval=1`, patience is counted by validation epochs/evaluations rather than mini-batches. One epoch is roughly 2.3k train images per fold, so this is the closest match to a ~2k-image no-improvement window.

Checkpoint safety remains on: `checkpoint_interval=1`, `auto_resume=True`, and all outputs are written under local `OUTPUT_ROOT / fold_model_tag`; best mAP@0.75 copies go to `BACKUP_DIR` with the fold model tag in the filename.


## Expected Outputs

- per-fold output folders under local `OUTPUT_ROOT`
- per-fold best checkpoint backups under `BACKUP_DIR`
- extracted source dataset: `DATASET_DIR`
- `FOLDED_DATASET_DIR / fold0..fold4 / {train,valid}`
- `FOLDED_DATASET_DIR / label_map_74.json`
- `FOLDED_DATASET_DIR / category_mapping.csv`
- `FOLDED_DATASET_DIR / _5fold_ready.json`
- fold summaries include `valid_canvas_images`, which should be `0` when `VALID_INCLUDE_CANVAS=False`

Training is launched with `python -u` and `PYTHONUNBUFFERED=1` so logs should stream during long runs. If the process stops, rerun the notebook. With an explicit `RESUME_CHECKPOINT_BY_FOLD` entry, that checkpoint is used first; otherwise `auto_resume: true` resumes from the latest `checkpoint_N.ckpt` in the local output folder. Finished folds with best checkpoints can be skipped manually by narrowing `FOLD_INDICES` before rerun.
